<a href="https://colab.research.google.com/github/harship123-source/Pfizer-Advanced-AI-Powered-Document-Insights-Data-Extraction-Externship/blob/main/Different_Open_Source_Embedding_Models_for_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ============================================
# COMPARING OPEN-SOURCE EMBEDDING MODELS FOR RAG
# ============================================


In [ ]:


!pip install -q llama-index llama-index-embeddings-huggingface pymupdf
!pip install -q nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 9.5 MB/s eta 0:00:00


In [ ]:
!pip install -q llama-index-llms-Groq

In [ ]:

import nest_asyncio
nest_asyncio.apply()

from llama_index.core import VectorStoreIndex, Document, Settings, get_response_synthesizer
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.query_engine import RetrieverQueryEngine
import fitz  # PyMuPDF
import time

# Disable LLM - we're only comparing embedding models for retrieval
from google.colab import userdata
from llama_index.llms.groq import Groq
import os


os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")


llm = Groq(
    model="qwen/qwen3.6-27b",
)

Settings.llm = llm

# ============================================
# SECTION 2: Load the Pharmaceutical Document
# ============================================



In [ ]:
from google.colab import files

uploaded = files.upload()

Saving sample-sdf-document.pdf to sample-sdf-document.pdf


In [ ]:


# Load the pharmaceutical SDF document
pdf_path = "/content/sample-sdf-document.pdf"
doc = fitz.open(pdf_path)
text = "\n".join([page.get_text() for page in doc])

print(f"Extracted {len(text.split())} words from the pharmaceutical document.")

Extracted 617 words from the pharmaceutical document.


In [ ]:


# Define a sentence splitter
text_splitter = SentenceSplitter(chunk_size=50, chunk_overlap=50)

# Turn raw text into a list of Document objects
documents = [Document(text=text)]

# Convert into nodes (smaller chunks)
nodes = text_splitter.get_nodes_from_documents(documents)

print(f"Created {len(nodes)} chunks from the document.")

Created 73 chunks from the document.


# ============================================
# SECTION 3: Compare Embedding Models
# ============================================
#


In [ ]:

embedding_models = {
    "MiniLM-L6-v2": "sentence-transformers/all-MiniLM-L6-v2",
    "BGE-small-en": "BAAI/bge-small-en-v1.5",
    "E5-small-v2": "intfloat/e5-small-v2"
}

# Pharmaceutical query about the SDF document
query = "What is the Product Release Criteria"

results = {}

for model_name, model_path in embedding_models.items():
    print(f"\n{'='*60}")
    print(f"Testing Embedding Model: {model_name}")
    print(f"{'='*60}")

    # Configure the embedding model
    embed_model = HuggingFaceEmbedding(model_name=model_path)
    Settings.embed_model = embed_model

    # Build a fresh index with this embedding model
    # This ensures documents are embedded with the same model as queries
    start_time = time.time()
    index = VectorStoreIndex(nodes)

    retriever = index.as_retriever(similarity_top_k=2)
    query_engine = RetrieverQueryEngine.from_args(retriever=retriever)

    # Run the query
    response = query_engine.query(query)
    end_time = time.time()

    # Retrieve nodes to store them for later display
    retrieved_nodes_for_model = retriever.retrieve(query)

    # Store results
    results[model_name] = {
        "response": str(response),
        "time": round(end_time - start_time, 2),
        "retrieved_nodes": retrieved_nodes_for_model # Store retrieved nodes here
    }

    print(f"Index build + retrieval time: {results[model_name]['time']} seconds")


Testing Embedding Model: MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Index build + retrieval time: 4.31 seconds

Testing Embedding Model: BGE-small-en


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Index build + retrieval time: 5.26 seconds

Testing Embedding Model: E5-small-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Index build + retrieval time: 15.92 seconds


In [ ]:

print("\n" + "="*70)
print("EMBEDDING MODEL COMPARISON RESULTS")
print("="*70)

for model, result in results.items():
    print(f"\n{'='*60}")
    print(f"Model: {model}")
    print(f"{'='*60}")
    print(f"Index Build + Retrieval Time: {result['time']} seconds")
    print(f"\nRetrieved Context:")
    print("-" * 40)
    print(result['response'])
    print()

    print(f"Raw Retrieved Nodes for {model}:")
    print("-" * 40)
    # Correctly display the retrieved nodes that were stored for this specific model
    for node_with_score in result['retrieved_nodes']:
      print(node_with_score.node.get_text())
    print()


print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print("\nModel Performance Ranking (by total time):")
sorted_results = sorted(results.items(), key=lambda x: x[1]['time'])
for i, (model, result) in enumerate(sorted_results, 1):
    print(f"  {i}. {model}: {result['time']}s")


EMBEDDING MODEL COMPARISON RESULTS

Model: MiniLM-L6-v2
Index Build + Retrieval Time: 4.31 seconds

Retrieved Context:
----------------------------------------

<think>
Thinking Process:
1.  **Analyze User Query:** The user asks "What is the Product Release Criteria" based on the provided context.
2.  **Analyze Context:**
   - Context contains two entries, both labeled "Product Release Criteria".
   - Text 1: "We hereby certify that the defined product has been manufactured to meet its specifications and have been verified to meet predetermined Critical to Quality attributes and the flow kit has been formally released for delivery."
   - Text 2: "We hereby certify that the defined product has been manufactured to meet its specifications and have been verified to meet predetermined Critical to Quality attributes and the flow kit has been formally released for" (cuts off, but essentially the same).
3.  **Extract Answer:** The Product Release Criteria states: "We hereby certify that the 